# SpecDrift-Bench — pilote v2 sur GPU T4 x2 (gratuit, sans TPU)

Réglages : **Accelerator : GPU T4 x2**, **Internet : On**.

La cellule « Serveur » essaie les modèles de la liste **dans l'ordre** et garde le premier qui démarre :
1. Qwen3.8-27B (AWQ 4 bits, version communautaire) : le plus fort, mais pas encore testé sur T4.
2. Qwen2.5-Coder-32B-Instruct (AWQ officielle) : même architecture que le 14B qui fonctionnait déjà sur T4.

Ensuite, on fait un **test de 2 tours** avant de lancer le pilote.

In [ ]:
!git clone -q https://github.com/roanenicki/specdrift-bench.git
%cd specdrift-bench
!pip install -q mini-swe-agent pyyaml pytest vllm
!pip uninstall -y -q torchaudio  # le torchaudio de Kaggle fait planter vLLM (versions CUDA différentes)
!python -m pytest -q tests  # doit afficher: 37 passed

## 1. Serveur : premier modèle qui démarre

In [ ]:
import subprocess, time, urllib.request, os, json

CANDIDATES = [
    # (modèle Hugging Face, contexte max, désactiver le mode "thinking" de Qwen3)
    ("barrydeen/Qwen3.8-27B-AWQ-4bit", 65536, True),
    ("barrydeen/Qwen3.8-27B-AWQ-4bit", 32768, True),
    ("Qwen/Qwen2.5-Coder-32B-Instruct-AWQ", 32768, False),
]

def start(model, max_len):
    log = open("/kaggle/working/vllm.log", "a")
    log.write(f"\n===== {model} max_len={max_len} =====\n"); log.flush()
    p = subprocess.Popen(
        ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", model,
         "--dtype", "float16", "--tensor-parallel-size", "2", "--max-model-len", str(max_len),
         "--gpu-memory-utilization", "0.92", "--port", "8000"],
        stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):  # jusqu'à 30 min (téléchargement compris)
        if p.poll() is not None:
            return None  # le serveur a planté : on passe au candidat suivant
        try:
            urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
            return p
        except Exception:
            time.sleep(10)
    p.kill()
    return None

server = None
for model, max_len, no_think in CANDIDATES:
    print(f"essai : {model} (contexte {max_len})", flush=True)
    server = start(model, max_len)
    if server:
        MODEL_HF, NO_THINK = model, no_think
        print(f"serveur prêt : {model}, contexte {max_len}")
        break
    print("  -> échec, voir /kaggle/working/vllm.log", flush=True)
    time.sleep(20)  # laisse les GPU se libérer
if not server:
    raise SystemExit("aucun modèle n'a démarré : envoie la fin de /kaggle/working/vllm.log")

MODEL = "hosted_vllm/" + MODEL_HF
EXTRA = ["--extra-body", json.dumps({"chat_template_kwargs": {"enable_thinking": False}})] if NO_THINK else []
os.environ["MSWEA_SILENT_STARTUP"] = "1"
os.environ["MSWEA_COST_TRACKING"] = "ignore_errors"

## 2. Test rapide : 2 tours

On veut voir `functional=pass` sur au moins un tour avant de lancer le pilote.

In [ ]:
subprocess.run(["python", "-m", "specdrift.runner", "--task", "tasks/t01_invlite", "--model", MODEL,
                "--api-base", "http://localhost:8000/v1", *EXTRA,
                "--condition", "continuous", "--n-turns", "2", "--out", "/kaggle/working/runs_smoke"])

## 3. Pilote : 3 conditions × 2 ordres, 10 tours chacun, 2 sessions à la fois

À lancer seulement si le test rapide a réussi au moins une demande.
Pour ne pas dépendre du navigateur : **Save Version → Save & Run All (Commit)** (12 h maximum).

In [ ]:
import glob
# Garde-fou : en mode "Save & Run All", le pilote ne démarre que si le test rapide a réussi
# au moins une demande. Sinon on s'arrête ici, sans consommer des heures de GPU pour rien.
smoke = [json.load(open(f)) for f in glob.glob("/kaggle/working/runs_smoke/**/results.json", recursive=True)]
ok = any(t.get("functional_pass") for r in smoke for t in r["turns"])
print("test rapide :", "au moins une demande réussie -> pilote lancé" if ok else "aucune demande réussie -> pilote annulé")

procs = []
jobs = [(row, cond) for row in [0, 1] for cond in ["continuous", "recall", "fresh"]] if ok else []
for i, (row, cond) in enumerate(jobs, start=1):
    log = open(f"/kaggle/working/pilot_row{row}_{cond}.log", "w")
    procs.append(subprocess.Popen(
        ["python", "-m", "specdrift.runner", "--task", "tasks/t01_invlite", "--model", MODEL,
         "--api-base", "http://localhost:8000/v1", *EXTRA, "--condition", cond,
         "--order-row", str(row), "--out", "/kaggle/working/runs"],
        stdout=log, stderr=subprocess.STDOUT))
    print(f"lancé : row {row} / {cond}", flush=True)
    if i % 2 == 0:
        for p in procs[-2:]:
            p.wait()
for p in procs:
    p.wait()
print("pilote terminé")

## 4. Résumé et export

In [ ]:
import glob
print("modèle :", MODEL_HF)
for f in sorted(glob.glob("/kaggle/working/runs*/**/results.json", recursive=True)):
    r = json.load(open(f)); s = r["summary"]
    print(f.split("working/")[1].rsplit("/", 1)[0],
          "| tours:", len(r["turns"]), "| fonctionnel:", s["functional_success_rate"],
          "| F (tours réussis):", s["mean_F_on_successful_turns"], "| fichiers cassés:", s["turns_with_broken_files"])
!cd /kaggle/working && zip -qr runs.zip runs runs_smoke vllm.log pilot_*.log 2>/dev/null; ls -lh /kaggle/working/runs.zip